# **응급상황 자동 인식 및 응급실 연계 서비스**
# **단계1 : 응급상황 음성 인식 및 요약**

## **0.미션**

단계 1에서는, 응급상황의 음성을 인식해서 텍스트로 변환하고, 변환된 텍스트를 다시 요약 및 핵심키워드 도출 작업을 수행합니다.  
이를 위해 사전학습된 모델을 API로 연결하여 활용합니다.

### (1) 미션1
* 음성인식 : STT(Speech-to-Text)
    * 사용 모델 : OpenAI의 **Whisper-1**
    * 제공받은 음성 파일과 새로 제작하는 5건 이상의 음성파일을 텍스트로 변환하고, 변환작업이 잘 되는지 확인해 봅시다.

### (2) 미션2
* 텍스트 요약 및 핵심 키워드 도출
    * 사용 모델 : OpenAI의 **GPT-3.5-turbo**
    * 내용 요약과 주요 키워드를 도출하도록
    프롬프트 입력과 출력을 구성하고 테스트 해 봅시다.

* [추가]응급실 현황 다운로드(이 데이터는 단계3에서 필요합니다.)



## **1.환경설정**

### (1) 경로 설정

구글 드라이브 연결

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(project6_2)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/project6_2/'

### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

* requirements.txt 파일의 [경로 복사]를 한 후,
* 아래 경로에 붙여 넣기

In [ ]:
# 경로 : /content/drive/MyDrive/project6_2/requirements.txt
# 경로가 다른 경우 아래 코드의 경로 부분을 수정하세요.

!pip install -r /content/drive/MyDrive/project6_2/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


#### 2) 라이브러리 로딩

In [ ]:
#필요한 라이브러리 설치 및 불러우기
import os
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
import openai
from openai import OpenAI
import json

# 더 필요한 라이브러리 추가 -------------




### (3) OpenAI API Key 환경 변수 설정

* 제공받은 open ai api key를 **api_key.txt** 파일에 저장합니다.
    * (제공받은 api_key.txt 파일은 비어 있습니다.)

* 다음 코드를 통해 환경변수로 등록 합니다.

In [ ]:
def load_file(filepath):
    with open(filepath, 'r') as file:
        return file.readline().strip()

# API 키 로드 및 환경변수 설정
openai.api_key = load_file(path + 'api_key.txt')
os.environ['OPENAI_API_KEY'] = openai.api_key

* ⚠️ 아래 코드셀은, 실행해서 key가 제대로 보이는지 확인하고 결과는 삭제하세요.

In [ ]:
print(os.environ['OPENAI_API_KEY'])

## **2. 미션1 : STT**

### (1) 제공된 데이터 변환
* 세부사항
    * 사용 모델 : whisper-1
    * 제공 받은 오디오 파일을 읽어서 텍스트로 변환시켜 봅시다.
        * 반복문을 통해 파일 하나씩 읽어서 텍스트 변환
        * 변환된 텍스트를 데이터 프레임에 추가

|filename|text|
|----|----|
|audio3.mp3|어쩌구 저쩌구...급해요.|

* 음성파일 변환

In [ ]:
# 음성파일 경로 지정
audio_path = path + 'audio/'

In [ ]:
# OpenAI 클라이언트 생성
client = OpenAI()

In [ ]:
# 위스퍼 모델 사용 : 제공된 음성파일 중 1개를 텍스트로 변환해보기
filename = 'audio2.mp3'
audio_file = open(audio_path + filename, "rb")
transcript = client.audio.transcriptions.create(
    file=audio_file,
    model="whisper-1",
    language="ko",
    response_format="text",
)

print(transcript, type(transcript))

119죠. 제가 지금 열이 열이 올랐어요. 몇 도냐면은 38도 정도 돼요. 머리가 아프고 좀 띵한 것 같아요. 우한이 좀 들어요. 어떻게 해야 할까요?
 <class 'str'>


* 음성파일 변환 함수 생성

In [ ]:
def audio_to_text(audio_path, filename):
    # OpenAI 클라이언트 생성
    client = OpenAI()

    # 오디오 파일을 읽어서, 위스퍼를 사용한 변환
    audio_file = open(audio_path + filename, "rb")
    transcript = client.audio.transcriptions.create(
        file=audio_file,
        model="whisper-1",
        language="ko",
        response_format="text",
    )

    # 결과 반환
    return transcript

In [ ]:
# 음성파일 이름을 리스트에 담기
file_names = [f for f in os.listdir(audio_path) if os.path.isfile(os.path.join(audio_path, f))]
print(file_names)

['audio4.mp3', 'audio5.mp3', 'audio2.mp3', 'audio1.mp3', 'audio3.mp3']


In [ ]:
# 반복문을 통해, 파일 하나씩 읽어서 텍스트 변환, 변환된 텍스트를 데이터 프레임에 추가
data_list = []

# 빈 데이터프레임 선언
data = pd.DataFrame(columns=['filename', 'text'])

# 반복문 수행하면서 오디오 변환
for filename in file_names:
  transcript = audio_to_text(audio_path, filename)
  data_list.append({'filename': filename, 'text': transcript})
  print(filename, transcript)

# 데이터프레임으로 변환
data = pd.DataFrame(data_list)


# 데이터프레임 결과 조회
data.head()

audio4.mp3 아까 가다가 머리를 박았는데, 처음에는 괜찮다가, 지금 3시간 정도 지났는데, 머리가 어지럽고 속이 매스꺼워요. 어떻게 해야 할까요?

audio5.mp3 화장실에서 미끄러워서 엉덩방아를 찍었어요. 그러고 꼬리뼈가 계속 아파요. 점점 아픈 것 같은데 응급실을 가야 할까요?

audio2.mp3 119죠. 제가 지금 열이 열이 올랐어요. 몇 도냐면은 38도 정도 돼요. 머리가 아프고 좀 띵한 것 같아요. 우한이 좀 들어요. 어떻게 해야 할까요?

audio1.mp3 지금 아빠가 넘어졌어요. 머리에서 피가 나는데 숨은 쉬고 있어요. 지금 막 일어났어요. 근데 조금 어지럽다고 하네요. 네네 계단에서 굴렀어요. 지금은 물 마시고 있는데 이거 응급실로 가봐야 할까요? 피도 지금 머졌어요. 네네 나이는 마흔아홉 살 이세요. 어떻게 해야 할지 모르겠어요.

audio3.mp3 동생이 콩 가지고 놀다가 코에 들어가서 한쪽 코가 막혔어요. 아무리 빼보려 해도 안 빠져요. 어떻게 해야 할까요? 동생이 너무 힘들어 하네요.



,filename,text
0,audio4.mp3,"아까 가다가 머리를 박았는데, 처음에는 괜찮다가, 지금 3시간 정도 지났는데, 머리..."
1,audio5.mp3,화장실에서 미끄러워서 엉덩방아를 찍었어요. 그러고 꼬리뼈가 계속 아파요. 점점 아픈...
2,audio2.mp3,119죠. 제가 지금 열이 열이 올랐어요. 몇 도냐면은 38도 정도 돼요. 머리가 ...
3,audio1.mp3,지금 아빠가 넘어졌어요. 머리에서 피가 나는데 숨은 쉬고 있어요. 지금 막 일어났어...
4,audio3.mp3,동생이 콩 가지고 놀다가 코에 들어가서 한쪽 코가 막혔어요. 아무리 빼보려 해도 안...


### (2) 오디오 데이터 추가 수집(제작) 및 변환

* 세부사항
    * 응급 상황에 맞는 음성 녹음하기
        * 응급 등급별 1개 이상씩(총 5개 이상)
    * 반복문을 통해 모든 음성 파일 데이터 변환 : STT
        * 변환 내용은 위에서 저장한 데이터프레임에 추가
    * 변환 후 음성 내용과 변환 결과를 비교


In [ ]:
audio_path_dh = path + 'audio_dh/'

In [ ]:
file_names_dh = [f for f in os.listdir(audio_path_dh) if os.path.isfile(os.path.join(audio_path_dh, f))]

In [ ]:
# 반복문을 통해, 파일 하나씩 읽어서 텍스트 변환, 변환된 텍스트를 데이터 프레임에 추가
data_list = []

# 빈 데이터프레임 선언
data_dh = pd.DataFrame(columns=['filename', 'text'])

# 반복문 수행하면서 오디오 변환
for filename in file_names_dh:
  transcript = audio_to_text(audio_path_dh, filename)
  data_list.append({'filename': filename, 'text': transcript})
  print(filename, transcript)

# 데이터프레임으로 변환
data_dh = pd.DataFrame(data_list)


# 데이터프레임 결과 조회
data_dh.head()

audio1_1.m4a 사람이 숨을 쉬지 않습니다. 갑자기 쓰러졌어요. 지금 심장이 멈춘 것 같아요. 어떻게 하나요?

audio1_2.m4a 사람이 심하게 다쳐서 피가 멈추지 않아요. 상처에서 계속 피가 흐르고 있어요. 빨리 도와주세요.

audio2_1.m4a 높은 곳에서 떨어져서 다리가 부러진 것 같아요. 움직일 때마다 너무 아파합니다. 어떻게 해야 할지 모르겠어요.

audio2_2.m4a 뜨거운 물에 아이가 화상을 입었어요. 피부가 빨갛게 부어올랐고 울고 있어요. 지금 어떻게 해야 하나요?

audio3_1.m4a 제 동생이 갑자기 배가 아프다고 누워서 움직이지 못하고 있어요. 너무 고통스러워 보입니다. 어떻게 해야 할까요?

audio3_2.m4a 제 형이 갑자기 어지럽다고 하더니 쓰러졌어요. 지금 숨은 쉬고 있는데 의식이 혼미해 보입니다.

audio4_1.m4a 사람이 발목을 삐었어요. 발목이 부어오르고 움직일때마다 많이 아파합니다. 지금은 얼음찜질을 하고 있습니다.

audio4_2.m4a 아이가 넘어져서 팔이 깊은 상처가 생겼어요. 상처에서 피가 조금씩 흐르고 있어요. 압박은 하고 있는데 어떻게 할지 모르겠습니다.

audio5_1.m4a 아이한테서 코피가 나는데 계속 멈추지 않고 있습니다. 지금은 고개를 숙이고 코를 잡고 있는데 괜찮을까요?

audio5_2.m4a 아이가 넘어져서 무릎에 찰과상을 입었어요. 물로 씻었는데 상처가 조금 깊은 것 같아요. 괜찮을까요?



,filename,text
0,audio1_1.m4a,사람이 숨을 쉬지 않습니다. 갑자기 쓰러졌어요. 지금 심장이 멈춘 것 같아요. 어떻...
1,audio1_2.m4a,사람이 심하게 다쳐서 피가 멈추지 않아요. 상처에서 계속 피가 흐르고 있어요. 빨리...
2,audio2_1.m4a,높은 곳에서 떨어져서 다리가 부러진 것 같아요. 움직일 때마다 너무 아파합니다. 어...
3,audio2_2.m4a,뜨거운 물에 아이가 화상을 입었어요. 피부가 빨갛게 부어올랐고 울고 있어요. 지금 ...
4,audio3_1.m4a,제 동생이 갑자기 배가 아프다고 누워서 움직이지 못하고 있어요. 너무 고통스러워 보...


In [ ]:
data = pd.concat([data, data_dh], ignore_index=True)

In [ ]:
data

,filename,text
0,audio4.mp3,"아까 가다가 머리를 박았는데, 처음에는 괜찮다가, 지금 3시간 정도 지났는데, 머리..."
1,audio5.mp3,화장실에서 미끄러워서 엉덩방아를 찍었어요. 그러고 꼬리뼈가 계속 아파요. 점점 아픈...
2,audio2.mp3,119죠. 제가 지금 열이 열이 올랐어요. 몇 도냐면은 38도 정도 돼요. 머리가 ...
3,audio1.mp3,지금 아빠가 넘어졌어요. 머리에서 피가 나는데 숨은 쉬고 있어요. 지금 막 일어났어...
4,audio3.mp3,동생이 콩 가지고 놀다가 코에 들어가서 한쪽 코가 막혔어요. 아무리 빼보려 해도 안...
5,audio1_1.m4a,사람이 숨을 쉬지 않습니다. 갑자기 쓰러졌어요. 지금 심장이 멈춘 것 같아요. 어떻...
6,audio1_2.m4a,사람이 심하게 다쳐서 피가 멈추지 않아요. 상처에서 계속 피가 흐르고 있어요. 빨리...
7,audio2_1.m4a,높은 곳에서 떨어져서 다리가 부러진 것 같아요. 움직일 때마다 너무 아파합니다. 어...
8,audio2_2.m4a,뜨거운 물에 아이가 화상을 입었어요. 피부가 빨갛게 부어올랐고 울고 있어요. 지금 ...
9,audio3_1.m4a,제 동생이 갑자기 배가 아프다고 누워서 움직이지 못하고 있어요. 너무 고통스러워 보...


## **3. 미션2 : Summary**

* 세부사항
    * 문서요약 예제 파일을 참조하여 테스트 해 봅니다.
    * 코드를 참조하여, 원하는 형식에 맞게 요약이 되도록 프롬프트를 구성합니다.
        * 요약 시 중요 키워드들이 함께 도출되도록 합니다.
        * 가능하다면, 요약 문장 길이에 제한을 둡시다.
    * 반복문을 통해 요약하고, 결과를 데이터프레임에 추가합니다.
        * summary 열을 추가하고, 요약 결과를 입력
            * 요약결과와 키워드는 하나의 문자열로 붙여서 summary열에 추가

### (1) 문서 요약

* 문서 요약 예제

In [ ]:
input_text = '''
한국은행 총재가 "올해 성장률이 기존 전망치 2.4%보다 낮아질 가능성이 크다"며 "2.2∼2.3% 정도로 떨어지지 않을까 생각한다"고 밝혔습니다.
이 총재는 오늘(29일) 국회 기획재정위원회 국정감사에 출석해 한은의 전망을 크게 밑돈 3분기 성장률을 바탕으로 올해 성장률 전망치가 조정될 가능성에 대해 이렇게 말했습니다.
성장률 하락의 가장 큰 요인인 수출 감소의 배경에 대해 이 총재는 "금액 기준으로 봐서는 수출이 안 떨어졌는데, 수량을 기준으로 떨어졌다"며 "자동차 파업 등 일시적 요인과 화학제품·반도체의 중국과 경쟁 등으로 수량이 안 늘어나는 것 같은데, 원인을 더 분석해봐야 할 사안"이라고 진단했습니다.
다음 달 28일 열릴 기준금리 결정 방향에 대해서는 "금리 결정할 때 하나의 변수만 보지 않고 종합적으로 보는데, 우선 미국 대선과 연방준비제도 금리 결정으로 경제 상황이 어떻게 변할지 보겠다"고 밝혔습니다.
또 "아울러 이후 달러가 어떻게 될지, 수출 등 내년 경제 전망과 거시안전성 정책이 부동산·가계부채에 미치는 영향 등도 고려해 결정하겠다"고 말했습니다.
'''

system_role = '''당신은 신문기사에서 핵심을 요약하는 어시스턴트입니다.
응답은 다음의 형식을 지켜주세요
{"summary": \"텍스트 요약\"}
'''

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": system_role
        },
        {
            "role": "user",
            "content": input_text
        }
    ]
)

# 답변
answer = response.choices[0].message.content
print(answer)

{"summary": "한국은행 총재는 올해 성장률이 2.4%보다 낮아질 가능성이 크다며 2.2∼2.3%로 예상하고, 수출 감소와 관련하여 자동차 파업과 중국과의 경쟁으로 수량이 감소한 것으로 분석했다. 또한 다음 달 28일 열리는 기준금리 결정 방향에 대해서는 미국 대선과 연방준비제도의 결정을 고려할 것이라고 밝혔다."}


* 문서 요약 함수로 생성

In [ ]:
def text_summary(input_text):
    # OpenAI 클라이언트 생성
    client = OpenAI()

    # 시스템 역할과 응답 형식 지정
    system_role = '''당신은 응급상황이 발생했을 때, 119 신고자의 전화 통화 내용을 요약하는 어시스턴트입니다.
    응답은 다음의 형식을 지켜주세요
    {"summary": \"텍스트 요약\"}
    '''

    # 입력데이터를 GPT-3.5-turbo에 전달하고 답변 받아오기
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": system_role
            },
            {
                "role": "user",
                "content": input_text
            }
        ]
    )

    # 응답 받기
    answer = response.choices[0].message.content

    # 응답형식을 정리하고 return
    return answer


* 저장된 text를 하나씩 불러와서 요약하고 다시 저장하기

In [ ]:
data_summary = data.copy()

for idx, row in data.iterrows():
  summary = text_summary(row['text'])
  data_summary.loc[idx, 'summary'] = summary

In [ ]:
data_summary['summary'][0]

'{"summary": "신고자는 머리를 부딪쳤고 지금 어지러움과 메스꺼움을 느낀다고 하며 3시간이 지났고 처음에는 괜찮았다고 한다. 응급상황일 수 있으므로 응급실을 찾아가는 것이 좋다고 안내해야 한다."}'

In [ ]:
data_summary

,filename,text,summary
0,audio4.mp3,"아까 가다가 머리를 박았는데, 처음에는 괜찮다가, 지금 3시간 정도 지났는데, 머리...","{""summary"": ""신고자는 머리를 부딪쳤고 지금 어지러움과 메스꺼움을 느낀다고..."
1,audio5.mp3,화장실에서 미끄러워서 엉덩방아를 찍었어요. 그러고 꼬리뼈가 계속 아파요. 점점 아픈...,"{""summary"": ""신고자는 화장실에서 미끄러져 엉덩방아를 다쳤고, 꼬리뼈가 계..."
2,audio2.mp3,119죠. 제가 지금 열이 열이 올랐어요. 몇 도냐면은 38도 정도 돼요. 머리가 ...,"{""summary"": ""응급상황 발생 시 119에 전화하여 열이 38도로 올랐고 머..."
3,audio1.mp3,지금 아빠가 넘어졌어요. 머리에서 피가 나는데 숨은 쉬고 있어요. 지금 막 일어났어...,"{""summary"": ""아빠가 계단에서 넘어져 머리에서 피가 나면서 어지러워하는 상..."
4,audio3.mp3,동생이 콩 가지고 놀다가 코에 들어가서 한쪽 코가 막혔어요. 아무리 빼보려 해도 안...,"{""summary"": ""동생이 콩으로 한쪽 코가 막혀서 고통을 겪고 있습니다. 콩을..."
5,audio1_1.m4a,사람이 숨을 쉬지 않습니다. 갑자기 쓰러졌어요. 지금 심장이 멈춘 것 같아요. 어떻...,"{""summary"": ""응급상황 발생, 현재 심장이 멈춘 것으로 보임. 심폐소생술을..."
6,audio1_2.m4a,사람이 심하게 다쳐서 피가 멈추지 않아요. 상처에서 계속 피가 흐르고 있어요. 빨리...,"{""summary"": ""다친 사람이 피가 멈추지 않고 상처에서 계속 흐르고 있다고 ..."
7,audio2_1.m4a,높은 곳에서 떨어져서 다리가 부러진 것 같아요. 움직일 때마다 너무 아파합니다. 어...,"{""summary"": ""환자가 높은 곳에서 떨어져 다리를 부러뜨렸다고 하며 움직일 ..."
8,audio2_2.m4a,뜨거운 물에 아이가 화상을 입었어요. 피부가 빨갛게 부어올랐고 울고 있어요. 지금 ...,"{""summary"": ""아이가 뜨거운 물에 화상을 입어 피부가 빨갛게 부어올랐고 울..."
9,audio3_1.m4a,제 동생이 갑자기 배가 아프다고 누워서 움직이지 못하고 있어요. 너무 고통스러워 보...,"{""summary"": ""신고자의 동생이 배가 아프다고 고통을 호소하며 누워있으며 움..."


### (2) 전국 병원 응급실 정보 수집



#### 1) 인증키 발급

* 인증키 발급 절차
    * 1) data.go.kr 회원가입
    * 2) 국립중앙의료원_전국 응급의료기관 정보 조회 서비스
https://www.data.go.kr/data/15000563/openapi.do 로 이동
    * 3) 활용신청
        * 활용목적 : 기타(개인 학습 용도)
        * 상세 기능선택
            * 응급의료기관 목록정보 조회
            * 응급의료기관 위치정보 조회
            * 응급의료기관 기본정보 조회
    * 4) 인증키 확인
        * 마이페이지 > Open API > 활용신청현황
        * [승인] 국립중앙의료원_전국 응급의료기관 정보 조회 서비스
        * 일반 인증키(Decoding) 이용

#### 2) 데이터 수집

In [ ]:
# path 확인
path = '/content/drive/MyDrive/project6_2/'

In [ ]:
# 응급실 데이터 수집하기

url = 'http://apis.data.go.kr/B552657/ErmctInfoInqireService/getEgytListInfoInqire'
serviceKey = '' # 여러분의 일반 인증키(Decoding)


params = {
    'serviceKey': serviceKey,
    'pageNo': '1', 'numOfRows': '1000',  # 전체 응급실 수가 500여개 됨. 1000개면 충분
    'format': 'xml'
}

response = requests.get(url, params = params)

# 정상 수행 되었다면 200
print(response)

<Response [200]>


In [ ]:
response.text

'<OpenAPI_ServiceResponse>\n\t<cmmMsgHeader>\n\t\t<errMsg>SERVICE ERROR</errMsg>\n\t\t<returnAuthMsg>SERVICE_KEY_IS_NOT_REGISTERED_ERROR</returnAuthMsg>\n\t\t<returnReasonCode>30</returnReasonCode>\n\t</cmmMsgHeader>\n</OpenAPI_ServiceResponse>'

In [ ]:
# response xml에서 주요 정보 찾기
root = ET.fromstring(response.text)

data_list = []

for item in root. findall('.//item'):
    duty_name = item.findtext('dutyName')
    duty_addr = item.findtext('dutyAddr')
    # 필요한 정보 추가
    hpid = item.findtext('hpid') # 기관ID
    wgs84lon = item.findtext('wgs84Lon') # 병원 경도
    wgs84lat = item.findtext('wgs84Lat') # 병원 위도

    # 병원 전화번호 / 응급실 전화번호
    duty_tel1 = item.findtext('dutyTel1') # 대표 전화번호
    duty_tel3 = item.findtext('dutyTel3') # 응급실 전화번호


    # 빈 리스트 data에 딕시너리 형태({'칼럼이름':값, ...})로 저장(추가)
    data_list.append({
        'hpid': hpid,
        'duty_name': duty_name,
        'duty_addr': duty_addr,
        'wgs84lon': wgs84lon,
        'wgs84lat': wgs84lat,
        'duty_tel1': duty_tel1,
        'duty_tel3': duty_tel3,
    })

# 데이터프레임으로 변환
df_list = pd.DataFrame(data_list)

In [ ]:
df_list

,hpid,duty_name,duty_addr,wgs84lon,wgs84lat,duty_tel1,duty_tel3
0,A1700023,(의)내경의료재단울산제일병원,울산광역시 남구 남산로354번길 26 (신정동),129.30701143429678,35.54823820112527,052-220-3300,052-220-3334
1,A1200028,(의)서일의료재단기장병원,부산광역시 기장군 기장읍 대청로72번길 6,129.21649161387128,35.23602946449906,051-723-0171,051-723-2119
2,A1400008,(의)성세의료재단 뉴성민병원,"인천광역시 서구 칠천왕로33번길 17 (석남동, 신석로 70(석남1동, 성민병원))",126.669478649026,37.5089936801167,032-726-1000,032-726-1190
3,A2100019,(의)영문의료재단다보스병원,"경기도 용인시 처인구 백옥대로1082번길 18, 다보스종합병원 (김량장동)",127.21049868474802,37.234641294534285,031-8021-2114,031-8021-2130
4,A2100122,(의)효심의료재단용인서울병원,경기도 용인시 처인구 고림로 81 (고림동),127.2144914405,37.240316373,031-337-0114,031-336-0119
...,...,...,...,...,...,...,...
521,A2100015,효산의료재단안양샘병원,"경기도 안양시 만안구 삼덕로 9 (안양동, 안양샘병원)",126.92447734066778,37.39340413136221,031-467-9717,031-467-9119
522,A2100054,효산의료재단지샘병원,"경기도 군포시 군포로 591 (당동, (G샘병원)군포샘병원)",126.94735972779895,37.35864464913468,031-389-3000,031-389-3119
523,A1200048,효성시티병원,부산광역시 해운대구 해운대로 135 (재송동),129.12145899191526,35.18541271514478,051-709-3000,051-709-3119
524,A1403132,흑룡의원,인천광역시 옹진군 백령면 백령로 831,124.6654985764,37.959524356,032-837-6873,032-837-3153


In [ ]:
df_list.to_csv(path + 'emergency_list.csv', index=False)

# 응급실 기본정보

In [ ]:
ID = []
for data_ in data_list:
  ID.append(data_['hpid'])

In [ ]:
# path 확인
path = '/content/drive/MyDrive/project6_2/'

In [ ]:
# 응급실 데이터 수집하기

url = 'http://apis.data.go.kr/B552657/ErmctInfoInqireService/getEgytBassInfoInqire'
serviceKey = '' # 여러분의 일반 인증키(Decoding)

data = []

for id in ID:
  params = {
      'serviceKey': serviceKey,
      'HPID': id,
      'format': 'xml'
  }
  response = requests.get(url, params = params)
  print(response, " : ",id)

  root = ET.fromstring(response.text)

  for item in root. findall('.//item'):
      duty_name = item.findtext('dutyName')
      duty_addr = item.findtext('dutyAddr')
      # 필요한 정보 추가
      duty_eryn = item.findtext('dutyEryn') # 응급실 운영여부 1 : Yes / 2 : No
      wgs84lon = item.findtext('wgs84Lon') # 병원 경도
      wgs84lat = item.findtext('wgs84Lat') # 병원 위도

      # 병원 전화번호 / 응급실 전화번호
      duty_tel1 = item.findtext('dutyTel1') # 대표 전화번호
      duty_tel3 = item.findtext('dutyTel3') # 응급실 전화번호

      # 응급실 가능여부
      mkioskty25 = item.findtext('MKioskTy25')
      hperyn = item.findtext('hperyn') # 응급실 수용가능한 환자
      # 세부 응급실 가능여부
      mkioskty1 = item.findtext('MKioskTy1') # 뇌출혈수술
      mkioskty2 = item.findtext('MKioskTy2') # 뇌경색의재관류
      mkioskty3 = item.findtext('MKioskTy3') # 심근경색의재관류
      mkioskty4 = item.findtext('MKioskTy4') # 복부손상의수술
      mkioskty5 = item.findtext('MKioskTy5') # 사지접합의수술
      mkioskty6 = item.findtext('MKioskTy6') # 응급내시경
      mkioskty7 = item.findtext('MKioskTy7') # 응급투석
      mkioskty8 = item.findtext('MKioskTy8') # 조산산모
      mkioskty9 = item.findtext('MKioskTy9') # 정신질환자
      mkioskty10 = item.findtext('MKioskTy10') # 신생아
      mkioskty11 = item.findtext('MKioskTy11') # 중증화상

      # 빈 리스트 data에 딕시너리 형태({'칼럼이름':값, ...})로 저장(추가)
      data.append({
          'duty_name': duty_name,
          'duty_addr': duty_addr,
          'wgs84lon': wgs84lon,
          'wgs84lat': wgs84lat,
          'duty_eryn': duty_eryn,
          'duty_tel1': duty_tel1,
          'duty_tel3': duty_tel3,
          'mkioskty25': mkioskty25,
          'mkioskty1': mkioskty1,
          'mkioskty2': mkioskty2,
          'mkioskty3': mkioskty3,
          'mkioskty4': mkioskty4,
          'mkioskty5': mkioskty5,
          'mkioskty6': mkioskty6,
          'mkioskty7': mkioskty7,
          'mkioskty8': mkioskty8,
          'mkioskty9': mkioskty9,
          'mkioskty10': mkioskty10,
          'mkioskty11': mkioskty11
      })

# 정상 수행 되었다면 200
print(data)

<Response [200]>  :  A1700023
<Response [200]>  :  A1200028
<Response [200]>  :  A1400008
<Response [200]>  :  A2100019
<Response [200]>  :  A2100122
<Response [200]>  :  A1300081
<Response [200]>  :  A1500019
<Response [200]>  :  A1400015
<Response [200]>  :  A1400012
<Response [200]>  :  A2100052
<Response [200]>  :  A2100012
<Response [200]>  :  A1100011
<Response [200]>  :  A1121013
<Response [200]>  :  A2100040
<Response [200]>  :  A1100047
<Response [200]>  :  A1123234
<Response [200]>  :  A1100141
<Response [200]>  :  A2100029
<Response [200]>  :  A1300018
<Response [200]>  :  A1100076
<Response [200]>  :  A1100043
<Response [200]>  :  A2200008
<Response [200]>  :  A2800032
<Response [200]>  :  A1100006
<Response [200]>  :  A1121842
<Response [200]>  :  A1117285
<Response [200]>  :  A2200009
<Response [200]>  :  A2200011
<Response [200]>  :  A2200007
<Response [200]>  :  A2200012
<Response [200]>  :  A2200002
<Response [200]>  :  A2200015
<Response [200]>  :  A2803677
<Response 

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df

,duty_name,duty_addr,wgs84lon,wgs84lat,duty_eryn,duty_tel1,duty_tel3,mkioskty25,mkioskty1,mkioskty2,mkioskty3,mkioskty4,mkioskty5,mkioskty6,mkioskty7,mkioskty8,mkioskty9,mkioskty10,mkioskty11
0,(의)내경의료재단울산제일병원,울산광역시 남구 남산로354번길 26 (신정동),129.30701143429678,35.54823820112527,1,052-220-3300,052-220-3334,None,None,None,None,None,None,None,None,None,None,None,None
1,(의)서일의료재단기장병원,부산광역시 기장군 기장읍 대청로72번길 6,129.21649161387128,35.23602946449906,1,051-723-0171,051-723-2119,N,N,N,N,N,N,N,N,N,N,N,Y
2,(의)성세의료재단 뉴성민병원,"인천광역시 서구 칠천왕로33번길 17 (석남동, 신석로 70(석남1동, 성민병원))",126.669478649026,37.5089936801167,1,032-726-1000,032-726-1190,N,N,N,N,N,N,N,N,N,N,N,N
3,(의)영문의료재단다보스병원,"경기도 용인시 처인구 백옥대로1082번길 18, 다보스종합병원 (김량장동)",127.21049868474802,37.234641294534285,1,031-8021-2114,031-8021-2130,N,N,N,N,Y,N,N,Y,Y,Y,Y,Y
4,(의)효심의료재단용인서울병원,경기도 용인시 처인구 고림로 81 (고림동),127.2144914405,37.240316373,1,031-337-0114,031-336-0119,N,N,N,N,N,N,N,N,N,N,N,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
521,효산의료재단안양샘병원,"경기도 안양시 만안구 삼덕로 9 (안양동, 안양샘병원)",126.92447734066778,37.39340413136221,1,031-467-9717,031-467-9119,N,Y,Y,Y,Y,N,N,Y,Y,Y,N,Y
522,효산의료재단지샘병원,"경기도 군포시 군포로 591 (당동, (G샘병원)군포샘병원)",126.94735972779895,37.35864464913468,1,031-389-3000,031-389-3119,N,Y,N,Y,Y,N,N,Y,N,Y,N,Y
523,효성시티병원,부산광역시 해운대구 해운대로 135 (재송동),129.12145899191526,35.18541271514478,1,051-709-3000,051-709-3119,None,None,None,None,None,None,None,None,None,None,None,None
524,흑룡의원,인천광역시 옹진군 백령면 백령로 831,124.6654985764,37.959524356,1,032-837-6873,032-837-3153,None,None,None,None,None,None,None,None,None,None,None,None


In [ ]:
# csv 파일로 저장(인덱스 제외)
df.to_csv(path + 'emergency_room.csv', index=False)

## **Mission Complete!**

수고 많았습니다!